In [ ]:
import os
import torch
import pandas as pd
from tqdm.auto import tqdm
from huggingface_hub import login

# ── Hugging Face Authentication ───────────────────────────────────────────────
# Opsi 1: Kaggle Secrets (otomatis)
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token diambil dari Kaggle Secrets")
except Exception:
    # Opsi 2: Environment variable
    hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    # Opsi 3: Input manual (jalankan sekali)
    import getpass
    hf_token = getpass.getpass("Masukkan Hugging Face Token: ")

login(token=hf_token, add_to_git_credential=False)

print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i} : {p.name}  |  VRAM {p.total_memory/1e9:.1f} GB")

# Evaluasi Preprocessing Raw Report

## Prepare GPU

In [ ]:
"""
AUTO-SELECT GPU
===============
Memindai semua GPU yang tersedia via nvidia-smi, lalu memilih satu GPU
dengan VRAM bebas terbanyak dan utilisasi terendah.
CUDA_VISIBLE_DEVICES di-set SEBELUM torch di-import agar efektif.
"""

import os
import subprocess
import sys

# ──────────────────────────────────────────────────────────────
# KONFIGURASI: berapa GPU yang ingin dipakai (1 atau lebih)
# Ubah N_GPUS_TO_USE = 2 jika ingin pakai 2 GPU terbaik, dst.
# ──────────────────────────────────────────────────────────────
N_GPUS_TO_USE = 1   # ← ubah sesuai kebutuhan

# ── Query nvidia-smi ──────────────────────────────────────────
QUERY_FIELDS = "index,name,memory.free,memory.total,utilization.gpu"
try:
    result = subprocess.run(
        [
            "nvidia-smi",
            f"--query-gpu={QUERY_FIELDS}",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
except FileNotFoundError:
    sys.exit("ERROR: nvidia-smi tidak ditemukan. Pastikan driver NVIDIA terinstall.")
except subprocess.CalledProcessError as e:
    sys.exit(f"ERROR: nvidia-smi gagal: {e.stderr.strip()}")

# ── Parse output ─────────────────────────────────────────────
gpu_info = []
for line in result.stdout.strip().splitlines():
    parts = [p.strip() for p in line.split(",")]
    if len(parts) < 5:
        continue
    try:
        gpu_info.append({
            "index"    : int(parts[0]),
            "name"     : parts[1],
            "mem_free" : int(parts[2]),   # MiB
            "mem_total": int(parts[3]),   # MiB
            "util_gpu" : int(parts[4]),   # %
        })
    except ValueError:
        continue  # baris tidak dapat di-parse, lewati

if not gpu_info:
    sys.exit("ERROR: Tidak ada GPU yang terdeteksi dari nvidia-smi.")

# ── Print tabel semua GPU ─────────────────────────────────────
header = f"{'IDX':>3}  {'Name':<30}  {'Free MiB':>10}  {'Total MiB':>10}  {'Util%':>6}"
sep    = "=" * len(header)
print(sep)
print("  GPU INFO (nvidia-smi)")
print(sep)
print(f"  {header}")
print("-" * len(header))
for g in gpu_info:
    marker = " ◄" if g is sorted(gpu_info, key=lambda x: (-x["mem_free"], x["util_gpu"]))[0] else ""
    print(
        f"  {g['index']:>3}  {g['name']:<30}  "
        f"{g['mem_free']:>10,}  {g['mem_total']:>10,}  {g['util_gpu']:>5}%{marker}"
    )
print(sep)

# ── Pilih N GPU terbaik (VRAM bebas terbanyak, lalu util terendah) ────────
sorted_gpus  = sorted(gpu_info, key=lambda g: (-g["mem_free"], g["util_gpu"]))
selected     = sorted_gpus[:N_GPUS_TO_USE]
selected_ids = [str(g["index"]) for g in selected]

# *** WAJIB di-set SEBELUM import torch ***
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected_ids)

print(f"\nGPU terpilih (N_GPUS_TO_USE={N_GPUS_TO_USE}):")
for rank, g in enumerate(selected):
    print(
        f"  cuda:{rank} ← GPU {g['index']}  {g['name']}"
        f"  |  VRAM bebas: {g['mem_free']:,} MiB / {g['mem_total']:,} MiB"
        f"  |  Util: {g['util_gpu']}%"
    )
print(f"\nCUDA_VISIBLE_DEVICES = \"{os.environ['CUDA_VISIBLE_DEVICES']}\"")
print("OK — lanjutkan ke cell berikutnya (import torch).")

## 1. Konfigurasi & Persiapan Data

In [ ]:
from pathlib import Path

# ── Ganti path sesuai lingkungan ──────────────────────────────────────────────
CSV_PATH = Path("data_raw.csv")          # lokal
# CSV_PATH = Path("/kaggle/input/...path.../raw_text.csv")  # Kaggle

df = pd.read_csv(CSV_PATH)
print(f"Shape   : {df.shape}")
print(f"Columns : {df.columns.tolist()}")
df.head(5)

In [ ]:
# Cek nilai unik & contoh teks untuk mengenali pola inkonsistensi
print("=== Contoh FINDINGS ===")
for i, text in enumerate(df["expertise_text_finding"].dropna().head(5)):
    print(f"[{i+1}] {text}\n")

print("=== Contoh CONCLUSION ===")
for i, text in enumerate(df["expertise_text_conclusion"].dropna().head(5)):
    print(f"[{i+1}] {text}\n")

## 2. Load Model

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = "google/gemma-3-12b-it"

# ── Quantization config (aktifkan jika VRAM < 12 GB) ─────────────────────────
USE_4BIT = True

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
) if USE_4BIT else None

print(f"Memuat tokenizer: {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print(f"Memuat model: {MODEL_ID} ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if not USE_4BIT else None,
    attn_implementation="eager",   # kompatibel luas antar GPU
)
model.eval()
print("Model siap.")

In [ ]:
SYSTEM_PROMPT = """Kamu adalah asisten normalisasi teks laporan radiologi toraks dalam Bahasa Indonesia.
Tugasmu adalah memperbaiki dan menstandarkan teks sesuai aturan berikut:

ATURAN EJAAN & KONSISTENSI ISTILAH:
- Gunakan "infiltrat" (bukan "infiltrate")
- Gunakan "bronkovaskuler" (bukan "bronchovaskuler" / "bronkhovaskuler")
- Gunakan "trakea" (bukan "trachea")
- Gunakan "toraks" (bukan "thoraks" / "thorax")
- Gunakan "sinus kostofrenikus" (bukan "sinus costophrenicus" / "sinus costofrenikus")
- Gunakan "hemidiafragma" (tetap)
- Gunakan "hilus" (tetap)
- Gunakan "mediastinum" (tetap)
- Gunakan "aorta" (tetap)
- Gunakan "kardiomegali" (bukan "cardiomegali" / "Cardiomegali")
- Gunakan "efusi pleura" (bukan "efusi pleura" sudah benar, pastikan konsisten)
- Gunakan "atelektasis" (bukan "atelektase")
- Gunakan "emfisema" (tetap)
- Gunakan "fibrosis" (tetap)
- Gunakan "kavitas" (bukan "cavitas")
- Gunakan "konsolidasi" (tetap)
- Gunakan "nodul" (bukan "nodus" dalam konteks ini)
- Gunakan "pneumonia" (huruf kecil kecuali awal kalimat)
- Gunakan "tuberkulosis paru" atau "Tuberkulosis paru" (bukan "Tb paru")
- Gunakan "Tidak tampak" (bukan "tak tampak")

ATURAN SINGKATAN:
Semua singkatan berikut harus selalu ditulis dalam bentuk lengkap diikuti singkatan dalam kurung:
- Jangan mempertahankan bentuk singkat saja.
- CTR → Cardiothoracic Ratio (CTR)
- CTI → Cardiothoracic Ratio (CTR)
- DD/ atau DD → Differential Diagnosis
- PA → Postero-Anterior (PA), jika merujuk pada jenis/proyeksi pemeriksaan
- AP → Antero-Posterior (AP), jika merujuk pada jenis/proyeksi pemeriksaan
- Semua kemunculan singkatan yang ada dalam daftar harus diperluas menjadi bentuk lengkap diikuti singkatan dalam kurung.

ATURAN FORMAT:
- Setiap kalimat/temuan diakhiri dengan titik.
- Jika ada daftar diagnosis, pisahkan per baris dengan tanda "-" di awal.
- Kapitalisasi hanya di awal kalimat dan nama penyakit/kondisi klinis yang lazim ditulis kapital.
- Hapus spasi ganda.
- Koreksi tanda baca yang salah tanpa mengubah makna klinis.

PENTING:
- JANGAN membuat kalimat pembuka seperti "Berikut adalah teks yang telah dinormalisasi sesuai dengan aturan yang diberikan:"
- JANGAN menambah catatan atau informasi tambahan.
- JANGAN menambah informasi yang tidak ada di teks asli kecuali perbaikan teks yang tercantum dalam aturan.
- JANGAN menghapus temuan klinis.
"""


def build_prompt(field_type: str, text: str) -> list[dict]:
    """Membangun pesan chat untuk normalisasi satu teks."""
    user_msg = (
        f"Normalisasikan teks {field_type} laporan radiologi berikut sesuai aturan.\n\n"
        f"TEKS ASLI:\n{text}\n\n"
        f"TEKS YANG SUDAH DINORMALISASI:"
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]

print("Prompt template siap.")

#### Contoh Sample

In [ ]:
MAX_NEW_TOKENS = 512   # cukup panjang untuk findings yang detail

@torch.inference_mode()
def normalize_text(text: str, field_type: str = "expertise_text_finding") -> str:
    """
    Normalisasi satu teks menggunakan Gemma 3.

    Parameters
    ----------
    text : str
        Teks asli (findings atau conclusion) yang akan dinormalisasi.
    field_type : str
        Label untuk konteks prompt ("findings" atau "conclusion").

    Returns
    -------
    str
        Teks yang sudah dinormalisasi.
    """
    if pd.isna(text) or not str(text).strip():
        return text  # kembalikan apa adanya jika kosong

    messages = build_prompt(field_type, str(text).strip())

    # apply_chat_template di transformers terbaru mengembalikan BatchEncoding
    # (bukan raw tensor), sehingga harus di-unpack dengan return_dict=True
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    prompt_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,          # greedy decoding → hapus temperature/top_p/top_k
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Ambil hanya token yang digenerate (bukan prompt)
    generated = outputs[0][prompt_len:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return result


# ── Uji cepat dengan satu contoh ──────────────────────────────────────────────
sample_finding = df["expertise_text_finding"].iloc[34]
sample_conclusion = df["expertise_text_conclusion"].iloc[34]

print("=== INPUT ===")
print("Findings   :", sample_finding)
print("Conclusion :", sample_conclusion)

norm_finding = normalize_text(sample_finding, "expertise_text_finding")
norm_conclusion = normalize_text(sample_conclusion, "expertise_text_conclusion")

print("\n=== OUTPUT ===")
print("Findings   :", norm_finding)
print("Conclusion :", norm_conclusion)

## 3. Normalisasi Seluruh Raw Data

In [ ]:
def normalize_column(df: pd.DataFrame, col: str, field_type: str) -> pd.Series:
    """Normalisasi seluruh nilai pada kolom `col` dengan progress bar."""
    results = []
    for text in tqdm(df[col], desc=f"Normalizing [{col}]", unit="row"):
        results.append(normalize_text(text, field_type))
    return pd.Series(results, index=df.index)


# ── Normalisasi kolom findings ────────────────────────────────────────────────
df["findings_normalized"] = normalize_column(df, "expertise_text_finding", "expertise_text_conclusion")

# ── Normalisasi kolom conclusion ──────────────────────────────────────────────
df["conclusion_normalized"] = normalize_column(df, "expertise_text_conclusion", "expertise_text_conclusion")

print(f"\nSelesai! Total baris diproses: {len(df)}")
df[["image_file", "expertise_text_finding", "findings_normalized",
    "expertise_text_conclusion", "conclusion_normalized"]].head(3)

In [ ]:
def show_changes(df: pd.DataFrame, original_col: str, normalized_col: str, max_rows: int = 10):
    """Cetak baris yang teksnya berubah setelah normalisasi."""
    changed = df[df[original_col].str.strip() != df[normalized_col].str.strip()].copy()
    print(f"[{original_col}] Jumlah baris berubah: {len(changed)} / {len(df)}\n")
    for idx, row in changed.head(max_rows).iterrows():
        print(f"--- Baris {idx} ({row.get('image_file', '')}) ---")
        print(f"  SEBELUM : {row[original_col]}")
        print(f"  SESUDAH : {row[normalized_col]}")
        print()


print("=" * 70)
show_changes(df, "expertise_text_finding", "findings_normalized")
print("=" * 70)
show_changes(df, "expertise_text_conclusion", "conclusion_normalized")

## 4. Simpan Hasil

In [ ]:
OUTPUT_DIR = Path(".")

# 1. Versi lengkap (dengan kolom asli + normalized untuk audit)
out_full = OUTPUT_DIR / "clean_dataset.csv"
df.to_csv(out_full, index=False, encoding="utf-8-sig")
print(f"Tersimpan (lengkap)  : {out_full}")

# 2. Versi final: ganti kolom asli dengan hasil normalisasi
df_final = df.copy()
df_final["expertise_text_finding"]   = df_final["findings_normalized"]
df_final["expertise_text_conclusion"] = df_final["conclusion_normalized"]
df_final = df_final.drop(columns=["findings_normalized", "conclusion_normalized"])

out_final = OUTPUT_DIR / "900_done_final.csv"
df_final.to_csv(out_final, index=False, encoding="utf-8-sig")
print(f"Tersimpan (final)    : {out_final}")

print("\nKolom pada file final:", df_final.columns.tolist())
df_final.head(3)

In [ ]:
def show_changes(df: pd.DataFrame, original_col: str, normalized_col: str, max_rows: int = 10):
    """Cetak baris yang teksnya berubah setelah normalisasi."""
    changed = df[df[original_col].str.strip() != df[normalized_col].str.strip()].copy()
    print(f"[{original_col}] Jumlah baris berubah: {len(changed)} / {len(df)}\n")
    for idx, row in changed.head(max_rows).iterrows():
        print(f"--- Baris {idx} ({row.get('image_file', '')}) ---")
        print(f"  SEBELUM : {row[original_col]}")
        print(f"  SESUDAH : {row[normalized_col]}")
        print()


print("=" * 70)
show_changes(df, "findings", "findings_normalized")
print("=" * 70)
show_changes(df, "conclusion", "conclusion_normalized")

## 5. Evaluasi

### WER (Word Error Rate)

In [ ]:
import pandas as pd
import re

def clean_text_for_evaluation(text):
    """
    Membersihkan teks khusus untuk perhitungan evaluasi modifikasi kata.
    Mengubah teks menjadi huruf kecil dan menghapus semua tanda baca.
    """
    if pd.isna(text):
        return ""
    
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    return " ".join(text.split())

def calculate_wer(reference, hypothesis):
    """
    Menghitung Word Error Rate (WER).
    """
    r_clean = clean_text_for_evaluation(reference)
    h_clean = clean_text_for_evaluation(hypothesis)
    
    r = r_clean.split()
    h = h_clean.split()
    
    d = [[0] * (len(h) + 1) for _ in range(len(r) + 1)]
    
    for i in range(len(r) + 1):
        d[i][0] = i
    for j in range(len(h) + 1):
        d[0][j] = j
        
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            if r[i - 1] == h[j - 1]:
                cost = 0
            else:
                cost = 1
            
            d[i][j] = min(
                d[i - 1][j] + 1,      # Deletion
                d[i][j - 1] + 1,      # Insertion
                d[i - 1][j - 1] + cost # Substitution
            )
                          
    if len(r) == 0:
        return float('inf') if len(h) > 0 else 0.0
        
    return d[len(r)][len(h)] / len(r)

def evaluate_dataset_vs_threshold(csv_file, threshold=0.16):
    """
    Menghitung rata-rata tingkat perubahan kata (WER) untuk seluruh dataset
    dan membandingkannya dengan ambang batas (tau).
    """
    print(f"Membaca dataset dari '{csv_file}'...")
    df = pd.read_csv(csv_file)
    df = df.fillna("")
    
    print("Menghitung Tingkat Perubahan (WER) per baris...")
    
    # Menggabungkan teks mentah (raw) dan teks bersih (clean)
    df['text_raw_combined'] = df['findings_raw'] + " " + df['conclusion_raw']
    df['text_clean_combined'] = df['findings_clean'] + " " + df['conclusion_clean']
    
    # Menghitung metrik WER pada setiap baris
    df['tingkat_perubahan_wer'] = df.apply(
        lambda row: calculate_wer(row['text_raw_combined'], row['text_clean_combined']), axis=1
    )
    
    # Menghitung Rata-Rata Keseluruhan
    mean_wer_overall = df['tingkat_perubahan_wer'].mean()
    total_data = len(df)
    
    # Evaluasi Dataset terhadap Ambang Batas
    memenuhi_threshold = mean_wer_overall <= threshold
    status_evaluasi = "MEMENUHI" if memenuhi_threshold else "TIDAK MEMENUHI"
    
    print("\n" + "="*55)
    print("HASIL EVALUASI TINGKAT PERUBAHAN DATASET (WER)")
    print("="*55)
    print(f"Total Laporan Dievaluasi     : {total_data} laporan")
    print(f"Rata-rata Tingkat Perubahan  : {mean_wer_overall:.4f} ({mean_wer_overall*100:.2f}%)")
    print(f"Ambang Batas (Tau)           : <= {threshold} ({threshold*100:.0f}%)")
    print("-" * 55)
    print(f"Status Evaluasi Sistem       : {status_evaluasi} TARGET")
    print("="*55 + "\n")
    
    # Membersihkan kolom bantuan
    df = df.drop(columns=['text_raw_combined', 'text_clean_combined'])
    
    return df, mean_wer_overall, memenuhi_threshold

# ==========================================
# Cara Penggunaan / Eksekusi
# ==========================================
if __name__ == "__main__":
    FILE_PATH = 'normalized_dataset.csv' 
    TAU_THRESHOLD = 0.16  # Anda dapat mengubah angka ini sesuai standar toleransi yang Anda mau
    
    try:
        df_result, mean_wer, is_passed = evaluate_dataset_vs_threshold(FILE_PATH, threshold=TAU_THRESHOLD)
        
        # Menyimpan DataFrame beserta nilai WER per baris jika ingin diperiksa lebih lanjut
        output_file = 'evaluasi_wer_dataset.csv'
        df_result.to_csv(output_file, index=False)
        print(f"Detail perhitungan per baris disimpan ke '{output_file}'")
        
    except FileNotFoundError:
        print(f"Error: File '{FILE_PATH}' tidak ditemukan.")

### BERTScore

In [ ]:
pip install bert-score pandas

In [ ]:
import pandas as pd
from bert_score import score
import warnings

# Mengabaikan peringatan yang tidak perlu dari transformers
warnings.filterwarnings('ignore')

def evaluate_with_bertscore(csv_file):
    """
    Mengevaluasi hasil normalisasi teks radiologi menggunakan BERTScore.
    Metrik ini mengukur kemiripan makna (semantik) antara teks asli dan normalisasi.
    """
    print(f"Membaca dataset dari '{csv_file}'...")
    df = pd.read_csv(csv_file)
    df = df.fillna("")
    
    # Menggabungkan teks mentah (raw) dan teks bersih (clean) menjadi satu dokumen utuh
    df['text_raw_combined'] = df['findings_raw'] + " " + df['conclusion_raw']
    df['text_clean_combined'] = df['findings_clean'] + " " + df['conclusion_clean']
    
    # Menyiapkan list untuk input ke BERTScore
    # references = Teks Asli (Raw)
    # candidates = Teks Normalisasi (Clean)
    references = df['text_raw_combined'].tolist()
    candidates = df['text_clean_combined'].tolist()
    
    print("\nMenghitung BERTScore (Proses ini mungkin memakan waktu beberapa menit tergantung ukuran data)...")
    
    # Menghitung BERTScore
    # lang='id' akan otomatis menggunakan model multilingual-BERT yang mendukung Bahasa Indonesia
    # verbose=True untuk menampilkan progress bar
    P, R, F1 = score(
        cands=candidates, 
        refs=references, 
        lang="id", 
        verbose=True
    )
    
    # Menyimpan skor ke dalam DataFrame
    df['bert_precision'] = P.numpy()
    df['bert_recall'] = R.numpy()
    df['bert_f1'] = F1.numpy()
    
    # Menghitung rata-rata skor keseluruhan dataset
    mean_precision = df['bert_precision'].mean()
    mean_recall = df['bert_recall'].mean()
    mean_f1 = df['bert_f1'].mean()
    
    total_data = len(df)
    
    print("\n" + "="*55)
    print("HASIL EVALUASI SEMANTIK DENGAN BERTSCORE")
    print("="*55)
    print(f"Total Laporan Dievaluasi : {total_data} laporan")
    print("-" * 55)
    print(f"Rata-rata Precision      : {mean_precision:.4f} (Seberapa relevan teks normalisasi)")
    print(f"Rata-rata Recall         : {mean_recall:.4f} (Seberapa banyak makna asli yang dipertahankan)")
    print(f"Rata-rata F1-Score       : {mean_f1:.4f} (Skor keseimbangan/kemiripan keseluruhan)")
    print("="*55 + "\n")
    
    # Membersihkan kolom bantuan
    df = df.drop(columns=['text_raw_combined', 'text_clean_combined'])
    
    return df

# ==========================================
# Cara Penggunaan / Eksekusi
# ==========================================
if __name__ == "__main__":
    FILE_PATH = 'normalized_dataset.csv' 
    OUTPUT_FILE = 'evaluasi_bertscore_dataset.csv'
    
    try:
        # Jalankan evaluasi
        df_result = evaluate_with_bertscore(FILE_PATH)
        
        # Simpan hasil evaluasi
        df_result.to_csv(OUTPUT_FILE, index=False)
        print(f"Detail perhitungan BERTScore per baris telah disimpan ke '{OUTPUT_FILE}'")
        
        # (Opsional) Tampilkan 3 data dengan skor terendah untuk dianalisis
        print("\n3 Laporan dengan F1-Score BERT terendah (Untuk Pengecekan Halusinasi):")
        lowest_f1_df = df_result.sort_values(by='bert_f1', ascending=True).head(3)
        for i, (index, row) in enumerate(lowest_f1_df.iterrows(), 1):
            print(f"\n[Skor F1: {row['bert_f1']:.4f}]")
            print(f"Raw   : {row['findings_raw']} {row['conclusion_raw']}")
            print(f"Clean : {row['findings_clean']} {row['conclusion_clean']}")
            print("-" * 55)
            
    except FileNotFoundError:
        print(f"Error: File '{FILE_PATH}' tidak ditemukan.")
    except Exception as e:
        print(f"Terjadi kesalahan: {e}")

### Faithfulness Check

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import warnings

warnings.filterwarnings('ignore')

def evaluate_faithfulness_nli(csv_file, batch_size=16):
    """
    Mengevaluasi Faithfulness teks menggunakan model Natural Language Inference (NLI).
    Mengukur probabilitas bahwa teks normalisasi (hypothesis) selaras (entailment) 
    dengan teks asli (premise).
    """
    print(f"Membaca dataset dari '{csv_file}'...")
    df = pd.read_csv(csv_file)
    df = df.fillna("")
    
    # Menggabungkan teks mentah (Premise) dan teks normalisasi (Hypothesis)
    df['premise'] = df['findings_raw'] + " " + df['conclusion_raw']
    df['hypothesis'] = df['findings_clean'] + " " + df['conclusion_clean']
    
    print("\nMemuat model NLI Multilingual...")
    # Menggunakan XLM-RoBERTa yang dilatih pada dataset XNLI (mendukung Bahasa Indonesia)
    # Model ini cocok untuk mendeteksi Entailment, Neutral, atau Contradiction
    model_name = "joeddav/xlm-roberta-large-xnli"
    
    device = 0 if torch.cuda.is_available() else -1
    nli_pipeline = pipeline("text-classification", model=model_name, device=device)
    
    print("Mulai menghitung Faithfulness Score (Entailment)...")
    print("Proses ini menggunakan komputasi yang lumayan berat, mohon tunggu...")
    
    faithfulness_scores = []
    labels = []
    
    # Memproses pasangan teks
    for idx, row in df.iterrows():
        # Format input untuk model NLI: "premise </s></s> hypothesis"
        text_pair = f"{row['premise']} </s></s> {row['hypothesis']}"
        
        try:
            # Truncate ke batas maksimal token model (512) jika teks terlalu panjang
            result = nli_pipeline(text_pair, truncation=True, max_length=512)[0]
            
            label = result['label']
            score = result['score']
            
            # Jika labelnya 'entailment', maka score adalah probabilitas entailment.
            # Jika label lain yang menang, kita anggap entailment prob-nya rendah 
            # (bisa diekstrak logits aslinya untuk akurasi lebih tinggi, tapi ini cukup untuk estimasi).
            if label == 'entailment':
                faithfulness_prob = score
            elif label == 'contradiction':
                faithfulness_prob = 1.0 - score
            else: # neutral
                faithfulness_prob = 0.5 
                
            labels.append(label)
            faithfulness_scores.append(faithfulness_prob)
            
        except Exception as e:
            print(f"Error pada indeks {idx}: {e}")
            labels.append("error")
            faithfulness_scores.append(0.0)
            
        # Tampilkan progress setiap 500 data
        if (idx + 1) % 500 == 0:
            print(f"[{idx + 1}/{len(df)}] data diproses...")
            
    df['nli_label'] = labels
    df['faithfulness_score'] = faithfulness_scores
    
    # Ringkasan Evaluasi
    mean_faithfulness = df['faithfulness_score'].mean()
    entailment_count = (df['nli_label'] == 'entailment').sum()
    contradiction_count = (df['nli_label'] == 'contradiction').sum()
    total_data = len(df)
    
    print("\n" + "="*65)
    print("HASIL EVALUASI FAITHFULNESS (NLI)")
    print("="*65)
    print(f"Total Laporan Dievaluasi     : {total_data} laporan")
    print(f"Rata-rata Faithfulness Score : {mean_faithfulness:.4f} (Mendekati 1.0 = Sangat Setia)")
    print("-" * 65)
    print(f"Jumlah Teks Entailment (Aman): {entailment_count} laporan")
    print(f"Jumlah Teks Contradiction    : {contradiction_count} laporan (PERLU DICEK MANUAL!)")
    print("="*65 + "\n")
    
    df = df.drop(columns=['premise', 'hypothesis'])
    return df

# ==========================================
# Cara Penggunaan / Eksekusi
# ==========================================
if __name__ == "__main__":
    FILE_PATH = 'normalized_dataset.csv' 
    OUTPUT_FILE = 'evaluasi_faithfulness_nli.csv'
    
    try:
        df_result = evaluate_faithfulness_nli(FILE_PATH)
        df_result.to_csv(OUTPUT_FILE, index=False)
        print(f"Detail evaluasi Faithfulness per baris telah disimpan ke '{OUTPUT_FILE}'")
        
        # Ekstrak otomatis sampel yang terindikasi kontradiksi (halusinasi)
        df_contradiction = df_result[df_result['nli_label'] == 'contradiction']
        if not df_contradiction.empty:
            df_contradiction.to_csv('sampel_kontradiksi_kritis.csv', index=False)
            print(f"PERINGATAN: Ditemukan {len(df_contradiction)} laporan yang berpotensi mengubah makna klinis.")
            print("Sampel ini dipisahkan ke 'sampel_kontradiksi_kritis.csv' untuk audit manual.")
            
    except FileNotFoundError:
        print(f"Error: File '{FILE_PATH}' tidak ditemukan.")